# Go2 バックフリップ強化学習（mjlab）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/masato-ka/mini-pupper4-backflip/blob/main/notebooks/backflip_go2_colab.ipynb)

mini-pupper 4 のバックフリップ方策の予備検証として、Unitree Go2 のシミュレーションで学習する。タスクは `Mjlab-Backflip-Flat-Unitree-Go2`。仕様は `四足バックフリップ強化学習 実装仕様書(mjlab).md`、実装の経緯は `docs/notes.md` にある。

学習はローカルと同じ mjlab の `train` コマンドで行う。この Notebook は、その実行環境を Colab 上に用意するためのもの。

**事前準備:** メニューの「ランタイム」→「ランタイムのタイプを変更」で GPU（T4 以上）を選ぶ。MuJoCo Warp での学習には CUDA が必要。

In [ ]:
!nvidia-smi

## 1. 設定

ここの値だけを書き換えれば、以降のセルはそのまま動く。

- `NUM_ENVS`: 並列環境数。T4 では 2048 から始め、メモリに余裕があれば 4096 に上げる。
- `LOGGER`: `tensorboard` か `wandb`。wandb を使う場合は、セル 3 でログインする。
- `USE_DRIVE`: True にすると、ログとチェックポイントを Google Drive に保存する。セッションが切れても残り、再開に使える。

In [ ]:
REPO_URL = "https://github.com/masato-ka/mini-pupper4-backflip.git"
BRANCH = "main"
REPO_DIR = "/content/mini-pupper4-backflip"

TASK = "Mjlab-Backflip-Flat-Unitree-Go2"
NUM_ENVS = 2048
MAX_ITERATIONS = 3000
LOGGER = "tensorboard"  # or "wandb"
RUN_NAME = "colab"

USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/mini-pupper4-backflip"

## 2. インストール

リポジトリを clone して、編集可能モード（`pip install -e`）で入れる。依存関係（mjlab 1.6.0、MuJoCo Warp、RSL-RL）はまとめて入る。

- タスクは entry point（`mjlab.tasks`）で mjlab に登録されるので、`train` コマンドからそのまま見える。
- mjviser 0.0.17 以降は MuJoCo 4 系を引き込んで mjlab 1.6.0 を壊すため、このパッケージ側で `<0.0.17` に固定している（`docs/notes.md`）。

In [ ]:
import os

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}
!pip install -q -e .

In [ ]:
# Colab ではオフスクリーン描画に EGL を使う（GLFW は使えない）。
os.environ["MUJOCO_GL"] = "egl"

# GPU の確認: torch と MuJoCo Warp の両方が CUDA を使えること。
!nvidia-smi --query-gpu=name,driver_version,memory.used --format=csv
import torch, warp as wp
print("torch", torch.__version__, "built for CUDA", torch.version.cuda,
      "| available:", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())
print("CUDA_VISIBLE_DEVICES =", repr(os.environ.get("CUDA_VISIBLE_DEVICES")))
wp.init()
print("warp CUDA devices:", wp.get_cuda_device_count())
!pip list 2>/dev/null | grep -i -E "^torch |warp-lang|mujoco|mjlab"
assert torch.cuda.is_available() and wp.get_cuda_device_count() > 0, "CUDA が使えません。ランタイムが GPU か、torch の CUDA 版を確認してください"
!list-envs 2>/dev/null | grep -E "Backflip|Go2"


## 3. ログの保存先（Google Drive、任意）

`train` はカレントディレクトリの `logs/rsl_rl/<experiment>/<日時>_<run_name>/` に書く。`logs` を Drive へのシンボリックリンクにして、チェックポイントを Drive に残す。

wandb を使う場合は、このセルの最後の行のコメントを外してログインする。

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(f"{DRIVE_DIR}/logs", exist_ok=True)
    if not os.path.islink("logs"):
        !rm -rf logs
        !ln -s {DRIVE_DIR}/logs logs
!ls -la logs 2>/dev/null || echo "logs/ はローカルに作られます"

# import wandb; wandb.login()

## 4. 学習前の確認

学習なしで、状態機械（5 相の遷移）と環境が動くことを確かめる。

- `check_state_machine.py`: 胴体に弾道の速度を与えて人工的に宙返りさせ、相が Stand → … → Done と進むことを確認する。ロボットが自力で跳ぶわけではない。
- テスト（任意、CPU で 1〜2 分）: 観測の次元、報酬の符号、終了条件、カリキュラムを確認する。

In [ ]:
!python scripts/check_state_machine.py --assist-landing 2>&1 | grep -E "^(t=|phase|monotonic|final)"

In [ ]:
# 任意: テスト一式。
!pip install -q pytest
!python -m pytest -q tests 2>&1 | tail -3

## 5. スモーク学習

64 環境で 5 イテレーションだけ回し、GPU で学習が通ることを確かめる。

In [ ]:
!train {TASK} --env.scene.num-envs 64 --agent.max-iterations 5 --agent.logger tensorboard --agent.run-name smoke 2>&1 | grep -v "Module .* load on device" > /content/smoke.log; grep -E "Training with|Environment device" /content/smoke.log; tail -40 /content/smoke.log


## 6. TensorBoard

学習中に別セルから見る。主に見る値は次のとおり。

| タグ | 内容 |
|---|---|
| `Curriculum/flip/stage` | 現在のステージ（1〜4 = S1〜S4） |
| `Curriculum/flip/metric` | 進級判定に使う達成率（S1: 回転到達率、S2: Mid 成功率、S3: コマンド別成功率の最小値） |
| `Metrics/flip/rate/*` | Air・Land への到達率、成功率（コマンド別を含む） |
| `Metrics/flip/ev14/*` | 高さ誤差、着地時の θ 誤差、トルクと関節速度のピーク、着地加速度のピーク、XY ずれ、ヨー変化 |
| `Episode_Termination/*` | 終了の内訳（TM-1〜TM-9） |
| `Episode_Reward/*` | 項ごとの報酬（項名が相ごとに分かれている） |

学習の初期は、まず静止して立つことを覚える。トリガーは 0.3 s 静止して初めて受理される（EV-6）ため、それまではフリップ関連の値が 0 のままになる。

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/rsl_rl/go2_backflip

## 7. 本学習

`--video True` を付けると、`--video-interval` ステップごとに学習中の動画が `logs/.../videos/train/` に保存される。

カリキュラムは達成率に応じて自動で進む。デバッグでステージを固定したい場合は `--env.curriculum.flip.params.fixed-stage <0..3>`（0 = S1）を付ける。

In [ ]:
!train {TASK} --env.scene.num-envs {NUM_ENVS} --agent.max-iterations {MAX_ITERATIONS} --agent.logger {LOGGER} --agent.run-name {RUN_NAME} --video True --video-interval 20000 2>&1 | grep -v "Module .* load on device" | grep -E "Training with|Environment device|Learning iteration|Mean reward|Curriculum/flip/(stage|metric)|Metrics/flip/rate/success|ETA"

## 8. 学習の再開

Colab のセッションが切れたら、Drive に残ったチェックポイントから再開する。

- `LOAD_RUN`: 再開する実行のディレクトリ名（`logs/rsl_rl/go2_backflip/` の下）。
- `START_STAGE`: 中断時のステージ（TensorBoard の `Curriculum/flip/stage` − 1）。ステージはチェックポイントに保存されないため、手で指定する。

In [ ]:
!ls logs/rsl_rl/go2_backflip/
LOAD_RUN = ""      # 例: "2026-11-14_15-00-00_colab"
START_STAGE = 0    # 0 = S1, 1 = S2, 2 = S3, 3 = S4

In [ ]:
assert LOAD_RUN, "LOAD_RUN を設定してください"
!train {TASK} --env.scene.num-envs {NUM_ENVS} --agent.max-iterations {MAX_ITERATIONS} --agent.logger {LOGGER} --agent.run-name {RUN_NAME}-resume --agent.resume True --agent.load-run {LOAD_RUN} --env.curriculum.flip.params.start-stage {START_STAGE} 2>&1 | grep -v "Module .* load on device" | grep -E "Training with|Environment device|Learning iteration|Mean reward|Curriculum/flip/(stage|metric)|Metrics/flip/rate/success|ETA"

## 9. 学習済み方策の動画

mjlab の `play` は録画後に対話ビューアを開いて止まるため、Colab では `scripts/record_video.py` を使う。最終ステージの設定（DR とノイズなし）で 1 エピソードを走らせ、mp4 を保存し、相の遷移と終了理由を表示する。

In [ ]:
import glob

runs = sorted(glob.glob("logs/rsl_rl/go2_backflip/*/"))
assert runs, "チェックポイントがありません"
ckpts = sorted(glob.glob(runs[-1] + "model_*.pt"), key=lambda p: int(p.rsplit("_", 1)[1][:-3]))
CHECKPOINT = ckpts[-1]
print(CHECKPOINT)

In [ ]:
import mediapy as media

for cmd in ("low", "mid", "high"):
    out = f"/content/flip_{cmd}.mp4"
    !python scripts/record_video.py --checkpoint {CHECKPOINT} --command {cmd} --out {out} 2>&1 | grep -E "^t=|^ +theta|INFO\] [0-9]+ frames"
    media.show_video(media.read_video(out), fps=50, title=cmd)

## 10. 成果物のダウンロード

Drive を使っていない場合は、チェックポイントと ONNX をダウンロードしておく。ONNX は runner が保存のたびに実行ディレクトリへ書き出している。

In [ ]:
from google.colab import files

files.download(CHECKPOINT)
for onnx in glob.glob(runs[-1] + "*.onnx"):
    files.download(onnx)